<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/social_media/Topic_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing Libraries**

In [1]:
!pip install tweepy
!pip install gensim

In [2]:
# Importing Libraries - make sure the packages are installed
import os
import tweepy as tw
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import requests
from io import BytesIO

# NLP
from wordcloud import WordCloud, STOPWORDS
import string
import nltk
from nltk import bigrams
from pathlib import Path
import random

#nltk library
nltk.download('punkt_tab')
nltk.download('stopwords')

from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

from nltk.corpus import stopwords as st
from nltk.tokenize import word_tokenize, sent_tokenize
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from gensim import corpora, models



import warnings
warnings.filterwarnings("ignore")

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# **Loading Data**

In [ ]:
# Reading data
%%time
FILE_ID = "1lhoPhOLB4fm-IgXlKhZFfC0UiFfHSklZ"
xlsx_url = f"https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx"
r = requests.get(xlsx_url)
df = pd.read_excel(BytesIO(r.content), engine="openpyxl")


In [ ]:
df.head()

# **Data Preprocessing**

In [ ]:
df.columns

In [ ]:
# Convert 'Date' column to datetime format
df['tweet_date'] = pd.to_datetime(df['Date'], errors='coerce')

# June 2025 Tweets
df_final = df[(df['tweet_date'].dt.year == 2025) & (df['tweet_date'].dt.month == 6)]

In [ ]:

# Clean function
def clean_text(text):
  """Takes a tweet, cleans it, lowercase and tokenises it.
  Input : string (text)
  Output: string (text)"""
  text = re.sub(r"http\S+|www\S+|https\S+", '', text)
  text = re.sub(r'\W+', ' ', text)
  text = text.lower()
  tokens = word_tokenize(text)
  tokens = [word for word in tokens if word not in stopwords.words('english')]
  return tokens

# Creating tokens
df_final['tokens'] = df_final['Title'].apply(clean_text)

# Hashtags function
def extract_hashtags(text):
  """Takes a tweet,extracts # (hashtags) from it.
  Input : string (text)
  Output: string (text)"""
  return re.findall(r"#\w+", text)

#extract hashtags
df_final['hashtags'] = df_final['Title'].apply(extract_hashtags)

# **Modelling**

In [ ]:

# Bigrams function
def get_bigrams(tokens):
  """Takes a tokens,and create bigrams from it.
  Input : string (text)
  Output: string (text)"""

  return [' '.join(bg) for bg in bigrams(tokens)]

# create bigrams
df_final['bigrams'] = df_final['tokens'].apply(get_bigrams)

In [ ]:

# Combine tokens and bigrams
df_final['combined'] = df_final['tokens'] + df_final['bigrams']

# Create dictionary and corpus
dictionary = corpora.Dictionary(df_final['combined'])
corpus = [dictionary.doc2bow(text) for text in df_final['combined']]

# Apply LDA
lda_model = models.LdaModel(corpus, num_topics = 7, id2word = dictionary, passes = 10)